# OVRO-LWA source metacatalog

Identify sources in OVRO-LWA wide-field FITS images with **PyBDSF**, then fuse per-image catalogs into a **metacatalog** with one entry per unique sky position.

Each FITS file is associated with an **LST hour bin** (e.g. `01h`) and a **color band** (`Full`, `Red`, `Green`, or `Blue`). PyBDSF is run independently on every image.

**Workflow**
1. Discover FITS files and parse LST hour + band from filenames.
2. Run `bdsf.process_image` on each `(lst_hour, band)` image → per-image source catalogs.
3. **LST merge** (per band): cross-match detections across LST hours within each band. One row per source, covering the union of sky seen in any LST hour. Pick the detection whose peak flux is nearest the **median** flux over matching LST images; copy **all** properties from that row.
4. **Band merge** (sequential): cross-match Blue to Full → temporary metacatalog; then Green to that catalog; then Red. Each step uses beam-sized radii and median-flux picks. Unmatched sources from any band become their own row (`bands_present` lists contributing bands).

Set `REUSE_CACHED_CATALOGS = True` to load existing `sources_{lst}_{band}.csv` and/or `metacatalog_lst_{band}.csv` from `OUTPUT_DIR` instead of re-running PyBDSF or LST merge.

Reference conventions: `ovro-lwa-portal` (`fits_to_zarr_xradio.py`) and `image-plane-correction` (`source_detection.py`).

In [ ]:
from __future__ import annotations

import os
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator

import astropy.units as u
import bdsf
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table

# --- user configuration ---------------------------------------------------
FITS_ROOT = Path("/fast/claw")  # directory containing FITS images (searched recursively)
OUTPUT_DIR = Path("/fast/claw/metacatalog")  # where catalogs are written

# PyBDSF detection parameters (see image-plane-correction/source_detection.py)
BDSF_KW = dict(
    thresh="hard",
    thresh_isl=7.0,
    thresh_pix=4.0,
    atrous_do=False,
    psf_vary_do=False,
    quiet=True,
    ncores=16,
)

# Optional subset of LST hour bins (e.g. ["01h", "02h"]). None = all discovered.
LST_HOURS_OVERRIDE: list[str] | None = None
ASSOC_BANDS = ("Blue", "Green", "Red")

# When True, skip PyBDSF / LST merge if matching CSVs already exist in OUTPUT_DIR
REUSE_CACHED_CATALOGS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Filename parsing

Generalized parsing supports several OVRO-LWA naming conventions:

| Convention | Example | LST hour | Band |
|------------|---------|----------|------|
| Deep color products | `I_01h_deep_Taper_R0_Full.fits` | `01h` | `Full` |
| LST color-band (portal ingest) | `Blue_I_10min_..._20250508_LST22h_t0001.fits` | `22h` | `Blue` |
| Parent directory | `.../01h/.../image.fits` | `01h` | from filename prefix |

Bands are one of `Full`, `Blue`, `Green`, `Red`. LST hour bins (`00h`–`23h`) are taken from discovered FITS filenames and parent directories matching the `??h` pattern — no manual hour list is required.

In [ ]:
COLOR_BANDS = ("Full", "Blue", "Green", "Red")
HOUR_DIR_RE = re.compile(r"^(\d{2})h$", re.IGNORECASE)

# Deep wideband color products: I_01h_deep_Taper_R0_Full.fits
DEEP_COLOR_RE = re.compile(
    r"^I_(\d{2})h_.*_(Full|Blue|Green|Red)\.fits$",
    re.IGNORECASE,
)

# Portal lst-color products: Blue_I_..._20250508_LST22h_t0001.fits
LST_COLOR_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*_(\d{8})_LST(\d{1,2})h_(t\d+)\.fits$",
    re.IGNORECASE,
)

# Band prefix fallback: Blue_I_....fits (LST from directory or LSTnnh elsewhere in name)
BAND_PREFIX_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*\.fits$",
    re.IGNORECASE,
)
LST_IN_NAME_RE = re.compile(r"LST(\d{1,2})h", re.IGNORECASE)


@dataclass(frozen=True, slots=True)
class FitsMetadata:
    path: Path
    lst_hour: str  # e.g. "01h"
    band: str  # Full | Blue | Green | Red
    time_key: str | None = None  # optional lst-color time bin key


def _format_lst_hour(hour: int | str) -> str:
    return f"{int(hour):02d}h"


def _lst_from_parents(path: Path) -> str | None:
    for parent in path.parents:
        m = HOUR_DIR_RE.match(parent.name)
        if m:
            return _format_lst_hour(m.group(1))
    return None


def parse_fits_metadata(path: Path) -> FitsMetadata | None:
    """Return LST hour and color band for a FITS path, or None if unrecognized."""
    name = path.name

    m = DEEP_COLOR_RE.match(name)
    if m:
        return FitsMetadata(path=path, lst_hour=_format_lst_hour(m.group(1)), band=m.group(2).title())

    m = LST_COLOR_RE.match(name)
    if m:
        band, ymd, lst_h, t_bin = m.group(1), m.group(2), m.group(3), m.group(4)
        return FitsMetadata(
            path=path,
            lst_hour=_format_lst_hour(lst_h),
            band=band.title(),
            time_key=f"{ymd}_LST{_format_lst_hour(lst_h)}_{t_bin}",
        )

    m = BAND_PREFIX_RE.match(name)
    if m:
        band = m.group(1).title()
        lst_h = None
        m_lst = LST_IN_NAME_RE.search(name)
        if m_lst:
            lst_h = _format_lst_hour(m_lst.group(1))
        else:
            lst_h = _lst_from_parents(path)
        if lst_h is None:
            return None
        return FitsMetadata(path=path, lst_hour=lst_h, band=band)

    return None


def discover_fits_files(root: Path, *, patterns: Iterable[str] = ("*.fits",)) -> list[FitsMetadata]:
    """Recursively find FITS files and parse metadata."""
    found: list[FitsMetadata] = []
    for pattern in patterns:
        for path in sorted(root.rglob(pattern)):
            meta = parse_fits_metadata(path)
            if meta is not None:
                found.append(meta)
    return found


def lst_hours_from_discovery(found: Iterable[FitsMetadata]) -> list[str]:
    """Return sorted unique LST hours (00h–23h) present in discovered FITS."""
    hours: set[str] = set()
    for meta in found:
        m = HOUR_DIR_RE.match(meta.lst_hour)
        if m is None:
            continue
        hour = int(m.group(1))
        if 0 <= hour <= 23:
            hours.add(_format_lst_hour(hour))
    return sorted(hours)


def discovered_slots(found: Iterable[FitsMetadata]) -> dict[tuple[str, str], FitsMetadata]:
    """Map (lst_hour, band) → metadata for every discovered FITS."""
    return {(m.lst_hour, m.band): m for m in found}

def slot_glob_patterns(lst_hour: str, band: str) -> list[str]:
    patterns = [
        f"**/*_{lst_hour}_*_{band}.fits",
        f"**/I_{lst_hour}_*_{band}.fits",
        f"I_{lst_hour}_*_{band}.fits",
    ]
    if band != "Full":
        patterns.insert(0, f"**/{band}_I_*_LST{lst_hour[0:2]}h_*.fits")
        patterns.insert(1, f"**/{band}_I_*_LST{int(lst_hour[:-1])}h_*.fits")
    else:
        patterns.insert(0, f"**/Full_I_*_LST{lst_hour[0:2]}h_*.fits")
    return patterns


def resolve_fits_slot(root: Path, lst_hour: str, band: str) -> Path:
    """Resolve exactly one FITS path for an (lst_hour, band) slot."""
    for pattern in slot_glob_patterns(lst_hour, band):
        matches = sorted({p.resolve() for p in root.glob(pattern)})
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise FileNotFoundError(
                f"Ambiguous glob for ({lst_hour}, {band}): pattern {pattern!r} matched "
                f"{len(matches)} files: {[m.name for m in matches]}"
            )
    raise FileNotFoundError(f"No FITS found for ({lst_hour}, {band}) under {root}")


In [ ]:
fits_files = discover_fits_files(FITS_ROOT)
_discovered_lst_hours = lst_hours_from_discovery(fits_files)
LST_HOURS = list(LST_HOURS_OVERRIDE) if LST_HOURS_OVERRIDE is not None else _discovered_lst_hours
fits_by_slot = discovered_slots(fits_files)
summary = pd.DataFrame(
    {
        "path": [m.path.name for m in fits_files],
        "lst_hour": [m.lst_hour for m in fits_files],
        "band": [m.band for m in fits_files],
        "time_key": [m.time_key for m in fits_files],
    }
)
print(f"Found {len(fits_files)} FITS files under {FITS_ROOT}")
print(
    f"LST hours from discovery ({len(_discovered_lst_hours)}): "
    f"{', '.join(_discovered_lst_hours)}"
)
if LST_HOURS_OVERRIDE is not None:
    print(f"LST hours in use (override): {', '.join(LST_HOURS)}")
else:
    print(f"LST hours in use (all discovered): {', '.join(LST_HOURS)}")
summary.sort_values(["lst_hour", "band"]).reset_index(drop=True)


## PyBDSF source detection

For each `(lst_hour, band)` image we:
1. Read the FITS primary HDU.
2. Sanitize non-finite pixels and ensure frequency keywords PyBDSF expects (`RESTFREQ`).
3. Pass the HDU directly to `bdsf.process_image`.
4. Export the Gaussian list (`catalog_type='gaul'`) as an Astropy `Table`, then convert to a pandas `DataFrame` with provenance columns.

In [ ]:
GAUL_COLUMNS = [
    "RA",
    "DEC",
    "E_RA",
    "E_DEC",
    "Total_flux",
    "E_Total_flux",
    "Peak_flux",
    "E_Peak_flux",
    "Maj",
    "E_Maj",
    "Min",
    "E_Min",
    "PA",
    "E_PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
    "S_Code",
    "Gaus_id",
    "Isl_id",
    "Source_id",
]


def _restfreq_hz(header: fits.Header) -> float | None:
    for key in ("RESTFREQ", "RESTFRQ", "CRVAL3", "FREQ"):
        if key in header:
            try:
                return float(header[key])
            except (TypeError, ValueError):
                continue
    return None


def _prepare_hdu(path: Path) -> fits.PrimaryHDU:
    """Read FITS, squeeze to 2D, sanitize NaNs, and fix header for PyBDSF."""
    with fits.open(path, memmap=True) as hdul:
        hdu = hdul[0]
        data = np.squeeze(np.asarray(hdu.data, dtype=np.float32))
        if data.ndim != 2:
            raise ValueError(f"Expected 2D image in {path.name}, got shape {data.shape}")
        data = np.where(np.isfinite(data), data, 0.0)
        header = hdu.header.copy()
    rf = _restfreq_hz(header)
    if rf is not None:
        header["RESTFREQ"] = rf
        header["RESTFRQ"] = rf
    return fits.PrimaryHDU(data=data, header=header)


def _beam_from_header(header: fits.Header) -> tuple[float, float, float]:
    if "BMAJ" not in header or "BMIN" not in header:
        raise ValueError("FITS header missing BMAJ/BMIN beam keywords")
    return (
        float(header["BMAJ"]),
        float(header["BMIN"]),
        float(header.get("BPA", 0.0)),
    )


def empty_sources_dataframe(
    meta: FitsMetadata,
    *,
    bmaj: float,
    bmin: float,
    bpa: float,
) -> pd.DataFrame:
    """Return an empty per-image catalog with expected columns."""
    columns = GAUL_COLUMNS + ["lst_hour", "band", "source_file", "BMAJ", "BMIN", "BPA"]
    if meta.time_key is not None:
        columns.append("time_key")
    return pd.DataFrame(columns=columns)


def run_pybdsf_on_hdu(hdu: fits.PrimaryHDU, **process_kw) -> Table | None:
    """Run PyBDSF on an in-memory HDU and return the Gaussian catalog table."""
    beam = _beam_from_header(hdu.header)
    kw = dict(BDSF_KW)
    kw.update(process_kw)
    kw["beam"] = beam

    img = bdsf.process_image(hdu, **kw)

    with tempfile.NamedTemporaryFile(suffix=".gaul.fits", delete=False) as tmp:
        cat_path = tmp.name
    try:
        img.write_catalog(outfile=cat_path, format="fits", catalog_type="gaul", clobber=True)
        if not os.path.isfile(cat_path) or os.path.getsize(cat_path) == 0:
            return None
        return Table.read(cat_path)
    except Exception:
        return None
    finally:
        try:
            os.unlink(cat_path)
        except OSError:
            pass


def detect_sources(meta: FitsMetadata) -> pd.DataFrame:
    """Detect sources in one FITS image; return a catalog DataFrame."""
    hdu = _prepare_hdu(meta.path)
    bmaj, bmin, bpa = _beam_from_header(hdu.header)
    table = run_pybdsf_on_hdu(hdu)
    if table is None or len(table) == 0:
        return empty_sources_dataframe(meta, bmaj=bmaj, bmin=bmin, bpa=bpa)

    df = table.to_pandas()

    keep = [c for c in GAUL_COLUMNS if c in df.columns]
    df = df[keep].copy()
    df["lst_hour"] = meta.lst_hour
    df["band"] = meta.band
    df["source_file"] = meta.path.name
    if meta.time_key is not None:
        df["time_key"] = meta.time_key
    df["BMAJ"] = bmaj
    df["BMIN"] = bmin
    df["BPA"] = bpa
    return df


def sources_csv_path(lst_hour: str, band: str) -> Path:
    return OUTPUT_DIR / f"sources_{lst_hour}_{band}.csv"


def lst_merged_csv_path(band: str) -> Path:
    return OUTPUT_DIR / f"metacatalog_lst_{band}.csv"


def all_sources_cached() -> bool:
    return all(
        sources_csv_path(lst, band).is_file() for (lst, band) in sorted(fits_by_slot)
    )


def all_lst_merged_cached() -> bool:
    return all(lst_merged_csv_path(band).is_file() for band in COLOR_BANDS)


def _ensure_bmaj_column(df: pd.DataFrame, lst_hour: str, band: str) -> pd.DataFrame:
    """Backfill BMAJ/BMIN/BPA from the FITS header when loading legacy source CSVs."""
    if "BMAJ" in df.columns and df["BMAJ"].notna().any():
        return df
    meta = fits_by_slot[(lst_hour, band)]
    with fits.open(meta.path, memmap=True) as hdul:
        bmaj, bmin, bpa = _beam_from_header(hdul[0].header)
    out = df.copy()
    out["BMAJ"] = bmaj
    out["BMIN"] = bmin
    out["BPA"] = bpa
    return out


def load_sources_catalog(csv_path: Path, lst_hour: str, band: str) -> pd.DataFrame:
    """Load a per-image source catalog CSV, ensuring beam columns are present."""
    df = pd.read_csv(csv_path)
    df = _ensure_bmaj_column(df, lst_hour, band)
    return df


def load_per_image_catalogs_from_disk() -> dict[tuple[str, str], pd.DataFrame]:
    """Load all per-image catalogs from OUTPUT_DIR when every slot is cached."""
    catalogs: dict[tuple[str, str], pd.DataFrame] = {}
    for lst_hour, band in sorted(fits_by_slot):
        csv_path = sources_csv_path(lst_hour, band)
        if not csv_path.is_file():
            raise FileNotFoundError(f"Missing cached catalog: {csv_path}")
        catalogs[(lst_hour, band)] = load_sources_catalog(csv_path, lst_hour, band)
    return catalogs


def load_lst_merged_from_disk() -> dict[str, pd.DataFrame]:
    """Load LST-merged per-band catalogs from OUTPUT_DIR."""
    merged: dict[str, pd.DataFrame] = {}
    for band in COLOR_BANDS:
        csv_path = lst_merged_csv_path(band)
        if not csv_path.is_file():
            raise FileNotFoundError(f"Missing cached LST merge: {csv_path}")
        merged[band] = pd.read_csv(csv_path)
    return merged


In [ ]:
per_image_catalogs: dict[tuple[str, str], pd.DataFrame] = {}

for (lst_hour, band), meta in sorted(fits_by_slot.items()):
    key = (lst_hour, band)
    csv_path = sources_csv_path(lst_hour, band)

    if REUSE_CACHED_CATALOGS and csv_path.is_file():
        catalog = load_sources_catalog(csv_path, lst_hour, band)
        per_image_catalogs[key] = catalog
        print(f"Loaded: {csv_path.name}  (LST={lst_hour}, band={band}, n={len(catalog)})")
        continue

    print(f"PyBDSF: {meta.path.name}  (LST={lst_hour}, band={band})")
    try:
        catalog = detect_sources(meta)
    except Exception as exc:
        print(f"  -> skipped ({exc})")
        continue

    per_image_catalogs[key] = catalog
    catalog.to_csv(csv_path, index=False)
    if len(catalog):
        print(f"  -> {len(catalog)} sources written to {csv_path}")
    else:
        print(f"  -> no sources detected; empty catalog written to {csv_path}")

len(per_image_catalogs)

## Metacatalog fusion

Two stages:

1. **`merge_lst_metacatalog`** — within each band, fuse detections from all LST hours. Matching uses greedy beam-sized clustering with vectorized `SkyCoord.separation`. The representative row is the detection whose `Peak_flux` is closest to the median over the cluster.

2. **`build_global_metacatalog`** — fuse the per-band LST-merged catalogs. Full-band rows are masters; Blue/Green/Red associations use vectorized `SkyCoord.search_around_sky`, then per-pair beam-radius filtering. Unmatched band-only sources are appended with `origin_band` set.

In [ ]:
BAND_FIELDS = (
    "Peak_flux",
    "Total_flux",
    "RA",
    "DEC",
    "Maj",
    "Min",
    "PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
)


def _pick_median_flux_row(df: pd.DataFrame, flux_col: str = "Peak_flux") -> pd.Series:
    """Return the row whose flux is closest to the median over *df*."""
    flux = df[flux_col].to_numpy(dtype=float)
    finite = np.isfinite(flux)
    if not finite.any():
        return df.iloc[0]
    med = float(np.nanmedian(flux[finite]))
    idx = int(np.nanargmin(np.abs(flux - med)))
    return df.iloc[idx]


def _skycoord_from_columns(df: pd.DataFrame) -> SkyCoord:
    return SkyCoord(
        ra=df["RA"].to_numpy(dtype=float) * u.deg,
        dec=df["DEC"].to_numpy(dtype=float) * u.deg,
    )


def _associate_catalogs(
    base_df: pd.DataFrame,
    band_df: pd.DataFrame,
) -> tuple[dict[int, list[int]], set[int]]:
    """Vectorized base↔band matching within per-pair beam radii.

    Returns ``{base_iloc: [band_iloc, ...]}`` and the set of matched band ilocs.
    """
    if base_df.empty or band_df.empty:
        return {}, set()

    base_sc = _skycoord_from_columns(base_df)
    band_sc = _skycoord_from_columns(band_df)
    bmaj_base = base_df["BMAJ"].to_numpy(dtype=float)
    bmaj_band = band_df["BMAJ"].to_numpy(dtype=float)

    search_radius = float(max(bmaj_base.max(), bmaj_band.max())) * u.deg
    # Astropy returns (idx_searcharound, idx_self) = (band, base)
    idx_band, idx_base, sep2d, _ = base_sc.search_around_sky(band_sc, search_radius)
    if len(idx_base) == 0:
        return {}, set()

    sep_deg = sep2d.to(u.deg).value
    limits = np.maximum(bmaj_base[idx_base], bmaj_band[idx_band])
    keep = sep_deg <= limits
    idx_base = idx_base[keep]
    idx_band = idx_band[keep]

    hits_by_base: dict[int, list[int]] = {}
    matched: set[int] = set()
    for i, j in zip(idx_base.tolist(), idx_band.tolist(), strict=True):
        hits_by_base.setdefault(i, []).append(j)
        matched.add(j)
    return hits_by_base, matched


def _cluster_by_sky_position(
    df: pd.DataFrame,
    *,
    sort_col: str = "Peak_flux",
    bmaj_col: str = "BMAJ",
) -> list[pd.DataFrame]:
    """Greedy beam-sized clustering; return one member DataFrame per cluster."""
    if df.empty:
        return []

    work = df[np.isfinite(df["RA"]) & np.isfinite(df["DEC"])].copy()
    work = work.sort_values(sort_col, ascending=False, na_position="last")

    cluster_ra: list[float] = []
    cluster_dec: list[float] = []
    cluster_bmaj: list[float] = []
    cluster_members: list[list[dict]] = []

    for row in work.itertuples(index=False):
        rd = row._asdict()
        ra_row = float(rd["RA"])
        dec_row = float(rd["DEC"])
        bmaj_row = float(rd.get(bmaj_col, np.nan))
        if not np.isfinite(bmaj_row):
            bmaj_row = 0.0

        best_idx: int | None = None
        if cluster_ra:
            sc = SkyCoord(ra=ra_row * u.deg, dec=dec_row * u.deg)
            cluster_sc = SkyCoord(
                ra=np.asarray(cluster_ra, dtype=float) * u.deg,
                dec=np.asarray(cluster_dec, dtype=float) * u.deg,
            )
            seps = sc.separation(cluster_sc).deg
            radii = np.maximum(bmaj_row, np.asarray(cluster_bmaj, dtype=float))
            within = seps <= radii
            if within.any():
                candidates = np.where(within)[0]
                best_idx = int(candidates[np.argmin(seps[candidates])])

        if best_idx is None:
            cluster_ra.append(ra_row)
            cluster_dec.append(dec_row)
            cluster_bmaj.append(bmaj_row)
            cluster_members.append([rd])
        else:
            cluster_members[best_idx].append(rd)
            rep = _pick_median_flux_row(pd.DataFrame(cluster_members[best_idx]))
            cluster_ra[best_idx] = float(rep["RA"])
            cluster_dec[best_idx] = float(rep["DEC"])
            cluster_bmaj[best_idx] = max(cluster_bmaj[best_idx], bmaj_row)

    return [pd.DataFrame(members) for members in cluster_members]


def merge_lst_metacatalog(catalogs: Iterable[pd.DataFrame], *, band: str) -> pd.DataFrame:
    """Fuse per-LST detections within one band → one row per source."""
    combined = pd.concat(list(catalogs), ignore_index=True)
    if combined.empty:
        return pd.DataFrame()

    rows: list[dict] = []
    for members in _cluster_by_sky_position(combined):
        rep = _pick_median_flux_row(members)
        entry = rep.to_dict()
        entry["band"] = band
        entry["n_lst_contributions"] = len(members)
        entry["lst_hours"] = ",".join(sorted(members["lst_hour"].unique()))
        entry["representative_lst"] = rep["lst_hour"]
        rows.append(entry)

    meta = pd.DataFrame(rows)
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)


def _empty_band_cols() -> dict:
    out: dict = {}
    for band in ASSOC_BANDS:
        for field in BAND_FIELDS:
            out[f"{field}_{band}"] = np.nan
        out[f"n_assoc_{band}"] = 0
    return out


def _lst_meta_from_band_row(row: pd.Series) -> tuple[int, str, str]:
    return (
        int(row.get("n_lst_contributions", 1)),
        str(row.get("lst_hours", row.get("lst_hour", ""))),
        str(row.get("representative_lst", row.get("lst_hour", ""))),
    )


def _primary_fields_from_band(row: pd.Series) -> dict:
    return {
        "RA": row["RA"],
        "DEC": row["DEC"],
        "Peak_flux": row["Peak_flux"],
        "Total_flux": row["Total_flux"],
        "Maj": row["Maj"],
        "Min": row["Min"],
        "PA": row["PA"],
        "DC_Maj": row.get("DC_Maj", np.nan),
        "DC_Min": row.get("DC_Min", np.nan),
        "DC_PA": row.get("DC_PA", np.nan),
    }


def _attach_band_columns(entry: dict, band_row: pd.Series, band: str, n_assoc: int) -> None:
    entry[f"n_assoc_{band}"] = n_assoc
    for field in BAND_FIELDS:
        entry[f"{field}_{band}"] = band_row[field]
    entry[f"source_file_{band}"] = band_row.get("source_file", "")


def _update_bands_present(entry: dict, *bands: str) -> None:
    present = {b for b in str(entry.get("bands_present", "")).split(",") if b}
    present.update(bands)
    order = ("Full", "Blue", "Green", "Red")
    entry["bands_present"] = ",".join(b for b in order if b in present)


def _seed_row_from_band(band_row: pd.Series, band: str) -> dict:
    """One metacatalog row seeded from a single-band LST-merged detection."""
    n_lst, lst_hours, rep_lst = _lst_meta_from_band_row(band_row)
    entry = {
        "origin_band": band,
        "bands_present": band,
        **_primary_fields_from_band(band_row),
        "BMAJ_match": float(band_row["BMAJ"]),
        "n_lst_contributions": n_lst,
        "lst_hours": lst_hours,
        "representative_lst": rep_lst,
    }
    entry.update(_empty_band_cols())
    if band == "Full":
        entry["BMAJ_full"] = float(band_row["BMAJ"])
        entry["source_file_Full"] = band_row.get("source_file", "")
    else:
        entry["BMAJ_full"] = np.nan
        _attach_band_columns(entry, band_row, band, 1)
    return entry


def merge_full_and_blue(full_df: pd.DataFrame, blue_df: pd.DataFrame) -> pd.DataFrame:
    """Cross-match Blue onto Full; one row per deduplicated sky position."""
    full_df = full_df.reset_index(drop=True)
    blue_df = blue_df.reset_index(drop=True)
    hits, matched_blue = _associate_catalogs(full_df, blue_df)

    rows: list[dict] = []
    for i, frow in full_df.iterrows():
        entry = _seed_row_from_band(frow, "Full")
        blues = hits.get(i, [])
        if blues:
            sub = blue_df.iloc[blues]
            best = _pick_median_flux_row(sub)
            _attach_band_columns(entry, best, "Blue", len(blues))
            _update_bands_present(entry, "Full", "Blue")
            entry["BMAJ_match"] = max(float(entry["BMAJ_match"]), float(best["BMAJ"]))
        rows.append(entry)

    for j, brow in blue_df.iterrows():
        if j not in matched_blue:
            rows.append(_seed_row_from_band(brow, "Blue"))
    return pd.DataFrame(rows)


def associate_band_into_metacatalog(
    meta_df: pd.DataFrame,
    band_df: pd.DataFrame,
    band: str,
) -> pd.DataFrame:
    """Cross-match one color band onto the current metacatalog; append unmatched band rows."""
    band_df = band_df.reset_index(drop=True)
    if meta_df.empty:
        return pd.DataFrame([_seed_row_from_band(brow, band) for _, brow in band_df.iterrows()])

    meta_df = meta_df.reset_index(drop=True)
    match_base = meta_df[["RA", "DEC"]].copy()
    match_base["BMAJ"] = meta_df["BMAJ_match"].to_numpy(dtype=float)
    hits, matched_band = _associate_catalogs(match_base, band_df)

    rows: list[dict] = []
    for i, mrow in meta_df.iterrows():
        entry = mrow.to_dict()
        band_hits = hits.get(i, [])
        if band_hits:
            sub = band_df.iloc[band_hits]
            best = _pick_median_flux_row(sub)
            _attach_band_columns(entry, best, band, len(band_hits))
            _update_bands_present(entry, band)
            entry["BMAJ_match"] = max(float(entry["BMAJ_match"]), float(best["BMAJ"]))
        rows.append(entry)

    for j, brow in band_df.iterrows():
        if j not in matched_band:
            rows.append(_seed_row_from_band(brow, band))
    return pd.DataFrame(rows)


def build_global_metacatalog(lst_merged: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Fuse LST-merged per-band catalogs via sequential cross-matching."""
    temp = merge_full_and_blue(lst_merged["Full"], lst_merged["Blue"])
    for band in ("Green", "Red"):
        temp = associate_band_into_metacatalog(temp, lst_merged[band], band)
    meta = temp
    meta.insert(0, "meta_id", range(len(meta)))
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)

In [ ]:
lst_merged: dict[str, pd.DataFrame] = {}

if REUSE_CACHED_CATALOGS and all_lst_merged_cached():
    lst_merged = load_lst_merged_from_disk()
    for band in COLOR_BANDS:
        print(
            f"LST merge ({band}): loaded {len(lst_merged[band])} sources from {lst_merged_csv_path(band)}"
        )
else:
    if not per_image_catalogs and REUSE_CACHED_CATALOGS and all_sources_cached():
        per_image_catalogs.update(load_per_image_catalogs_from_disk())
        print(f"Loaded {len(per_image_catalogs)} per-image catalogs from {OUTPUT_DIR}")

    for band in COLOR_BANDS:
        band_catalogs = [
            per_image_catalogs[(lst, band)]
            for lst in LST_HOURS
            if (lst, band) in per_image_catalogs
        ]
        merged = merge_lst_metacatalog(band_catalogs, band=band)
        lst_merged[band] = merged
        out_csv = lst_merged_csv_path(band)
        merged.to_csv(out_csv, index=False)
        print(f"LST merge ({band}): {len(merged)} sources -> {out_csv}")

metacatalog = build_global_metacatalog(lst_merged)

meta_csv = OUTPUT_DIR / "metacatalog.csv"
meta_fits = OUTPUT_DIR / "metacatalog.fits"
metacatalog.to_csv(meta_csv, index=False)
Table.from_pandas(metacatalog).write(meta_fits, overwrite=True)

if per_image_catalogs:
    n_inputs = sum(len(df) for df in per_image_catalogs.values())
    input_desc = f"{n_inputs} per-image detections"
else:
    n_inputs = sum(int(df["n_lst_contributions"].sum()) for df in lst_merged.values())
    input_desc = f"{n_inputs} LST-merged rows (cached)"
print(f"\nGlobal metacatalog: {len(metacatalog)} sources from {input_desc}")
print(f"Wrote {meta_csv}")
print(f"Wrote {meta_fits}")
metacatalog.head(10)

In [ ]:
# LST merge yield (Full band)
full_lst = lst_merged["Full"]
multi_lst = full_lst[full_lst["n_lst_contributions"] > 1].sort_values("n_lst_contributions", ascending=False)
print(f"Full-band sources after LST merge: {len(full_lst)}")
print(f"  seen in multiple LST hours: {len(multi_lst)}")
if len(multi_lst):
    display(multi_lst.head(10)[["RA", "DEC", "Peak_flux", "n_lst_contributions", "lst_hours", "representative_lst"]])

# Global band merge
print(f"\nGlobal rows by origin_band:")
print(metacatalog["origin_band"].value_counts())

full_with_color = metacatalog[
    (metacatalog["origin_band"] == "Full")
    & (metacatalog[[f"n_assoc_{b}" for b in ASSOC_BANDS]].max(axis=1) > 0)
]
print(f"Full-seeded rows with at least one color-band association: {len(full_with_color)}")
metacatalog.head(10)[["meta_id", "RA", "DEC", "origin_band", "Peak_flux", "lst_hours", "n_assoc_Blue", "n_assoc_Green", "n_assoc_Red"]]

In [ ]:
metacatalog['RA'].hist()